# Learning Optimization Theory

##### What are the Components of a Solver?
1. Decision Variables: How many kW per hour should we charge?
2. Objective: We minimize for total bill (price x kWh) as a sum over hours we charge
3. Constraints: The rules we need to follow (Max 6kWh, need 80% by 7am, between 20% and 100%, can only charge when on charger)
Also: We need to keep track of the battery charge over time e.g. start_level, start_level + hour 1 charge, start_level + hour 2 charge, ... 

#### In a simple example:
1. Choices (Which numbers does the solver choose): charge_kw, one per hour (24 charges, one start is already given)
2. Objective: min: sum(price_h x kW_h)
3. Rules: For every hour:     
    Charger delivers max 6kW,     
    Car on charger = YES,     
    never below 20% or above 100%,    
    no negative charge,    
    next_level = this_level +charge * 0.95   
   For one specific hour: >=48kWh at 7:00    
   start at 18kWh,   

Coding:

In [1]:
import cvxpy as cp
import numpy as np

from bev.data import tou_prices_jpy_per_kwh


In [2]:
# Variables for the model

prices = tou_prices_jpy_per_kwh(24, start_hour=18)
at_departure = 13
plugged_in = np.array([1]*(at_departure) + [0]*(24-(at_departure)))
battery_size_kWh = 60
charger_power_kW = 6
charger_efficiency = 0.95
start_level_kWh = 18
minimum_level_kWh = 12
target_level_kWh = 100
charge_kW = cp.Variable(24)
soc_kWh = cp.Variable(25)

In [3]:
# Constraints for the model

constraints = [] 

constraints.append(soc_kWh[0] == 18) # start level is at 18kWh
constraints.append(charge_kW <= charger_power_kW * plugged_in) # we can only charge within the charger limit
constraints.append(soc_kWh >= minimum_level_kWh) # we need to have always more than the minimum level defined
constraints.append(soc_kWh <= battery_size_kWh) # we cannot exceed the batteries maximum
constraints.append(charge_kW >= 0) # charge never can be negative
constraints.append(soc_kWh[at_departure] >= target_level_kWh) # we need to reach the target level at departure time
for t in range(24):
    constraints.append(soc_kWh[t+1] == soc_kWh[t]+charge_kW[t]*charger_efficiency) # we define each soc (state of charge?) as a function


In [4]:
# Problem Solver

problem = cp.Problem(cp.Minimize(prices @ charge_kW), constraints)
problem.solve()
problem.status
#print(problem.status, problem.value, charge_kW.value.round(2), soc_kWh.value.round(2))

'infeasible'

#### Explainer:
The decision variable we solve for is energy drawn per hour. The objective is to minimize the sum of price x kwh (Total Bill).
As for the constraints, we define a rule that links the charge to the battery state * efficiency of the charger, as some energy is lost during the process. Also the car can only charge while plugged in, and must be at 80% by departure, as well as limits of the battery and the charger.

#### With written charging.py

In [5]:
from bev.charging import optimise_charging

In [6]:
prices = tou_prices_jpy_per_kwh(24, start_hour=18)
plugged_in = np.array([1]*(at_departure) + [0]*(24-(at_departure)))
at_departure = 13
start_level_kWh = 18

optimise_charging(price_jpy_per_kwh=prices, plugged_in=plugged_in, soc_init_kwh=start_level_kWh,departure_step=at_departure)

ChargingResult(charge_kw=array([ 6.14243651e-10,  1.48632275e-10,  1.87632546e-10,  3.71868367e-10,
        5.22923753e-10,  4.32960366e+00,  4.07824328e+00,  3.88104773e+00,
        3.75099909e+00,  3.70612387e+00,  3.75524617e+00,  3.90464673e+00,
        4.17303684e+00, -1.31597207e-10, -1.18710903e-10, -8.08832348e-11,
       -5.73994154e-11, -5.27485469e-11, -5.72042702e-11, -6.77172361e-11,
       -8.73881888e-11, -1.13215298e-10, -1.30097207e-10, -1.30185177e-10]), soc_kwh=array([18.        , 18.        , 18.        , 18.        , 18.        ,
       18.        , 22.11312348, 25.98745459, 29.67444994, 33.23789907,
       36.75871675, 40.32620061, 44.035615  , 48.        , 48.        ,
       48.        , 48.        , 48.        , 48.        , 48.        ,
       48.        , 48.        , 48.        , 48.        , 48.        ]), cost_jpy=np.float64(789.473684201314), status='optimal')